# Linear Regression – Concepts

**Slides:** Part 03 «Regression models» – linear regression.

This notebook summarises the concepts of the slides and adds explanations and small examples. All examples use the numbers of the slides or small simulated data – no data set is loaded.

**Contents**
1. The model
2. Ordinary least squares (OLS)
3. Worked example: rent of a flat
4. Goodness of fit: $R^2$ and RMSE
5. Confidence vs. prediction interval
6. Assumptions and diagnostics
7. Multiple predictors and *ceteris paribus*
8. Maximum likelihood
9. Gradient descent

## Libraries and settings

In [ ]:
# Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

## 1. The model

**Simple linear regression** (one feature):

$$y = \beta_0 + \beta_1 x + \varepsilon \qquad (\beta_0 \text{ intercept},\ \beta_1 \text{ slope},\ \varepsilon \text{ error term})$$

**Multiple regression** (several features):

$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots + \beta_p x_p + \varepsilon$$

In words: $y$ is a weighted sum of the features plus a random error $\varepsilon$; each $\beta_j$ is the change in the
**average** $y$ per unit of its feature, **holding the other features constant**.

**Key idea:** "linear" means linear in the **parameters** $\beta$. Transformed features such as $x^2$, $\log x$ or
products $x_1 \cdot x_2$ (interactions) are allowed – the model is still fitted by linear regression.

## 2. Ordinary least squares (OLS)

**Objective:** choose $\beta_0, \beta_1$ so that the sum of squared residuals is as small as possible:

$$\text{RSS}(\beta_0, \beta_1) = \sum_{i=1}^{n}\big(y_i - (\beta_0 + \beta_1 x_i)\big)^2 \;\rightarrow\; \min$$

**Solution (one predictor):**

$$\hat\beta_1 = \frac{\sum_i (x_i-\bar x)(y_i-\bar y)}{\sum_i (x_i-\bar x)^2}, \qquad \hat\beta_0 = \bar y - \hat\beta_1 \bar x$$

In words: the slope measures how strongly $x$ and $y$ move together, relative to the spread of $x$; the line always passes
through the point of means $(\bar x, \bar y)$. With several predictors the solution in matrix form is
$\hat\beta = (X^\top X)^{-1} X^\top y$.

In [ ]:
rng = np.random.default_rng(7)
x = rng.uniform(30, 140, 25)
y = 170 + 31 * x + rng.normal(0, 250, 25)

beta_1 = np.sum((x - x.mean()) * (y - y.mean())) / np.sum((x - x.mean())**2)
beta_0 = y.mean() - beta_1 * x.mean()
print(f'beta_0 = {beta_0:.2f}, beta_1 = {beta_1:.2f}')

plt.figure(figsize=(7, 4))
plt.scatter(x, y, color='black', s=15, zorder=3)
xx = np.array([25, 145])
plt.plot(xx, beta_0 + beta_1 * xx, color='tab:blue', label='OLS line')
plt.vlines(x, y, beta_0 + beta_1 * x, color='tab:orange', linewidth=1, label='residuals')
plt.scatter([x.mean()], [y.mean()], color='darkred', s=60, zorder=4, label='point of means')
plt.xlabel('living area [m²]')
plt.ylabel('rent [CHF/month]')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 3. Worked example: rent of a flat

On the slides, the OLS line for the rent of flats is $\widehat{\text{rent}} = 166.79 + 30.84 \cdot \text{area}$.

- $\hat\beta_1 = 30.84$: +1 m² → +30.84 CHF per month (on average).
- $\hat\beta_0 = 166.79$: value at $x = 0$ – outside the data, no substantive meaning.
- Prediction for a 95 m² flat: $166.79 + 30.84 \cdot 95 = 3'096.59$ CHF.
- Extrapolation beyond the observed range of $x$ is risky.

In [ ]:
beta_0, beta_1 = 166.79, 30.84
print(f'Predicted rent for 95 m2: {beta_0 + beta_1 * 95:.2f} CHF')

## 4. Goodness of fit: $R^2$ and RMSE

For OLS with an intercept, the total variation of $y$ splits into an explained part and an unexplained rest:

$$\underbrace{\sum_i (y_i-\bar y)^2}_{\text{SST}} = \underbrace{\sum_i (\hat y_i-\bar y)^2}_{\text{SSR (explained)}}
+ \underbrace{\sum_i (y_i-\hat y_i)^2}_{\text{SSE (unexplained)}}$$

$$R^2 = 1 - \frac{\text{SSE}}{\text{SST}} \qquad \text{RMSE} = \sqrt{\frac{1}{n}\sum_i (y_i - \hat y_i)^2}$$

- $R^2$ = share of the variation explained (0 = nothing, 1 = perfect fit). Example: SSE = 20, SST = 100 → $R^2 = 0.8$.
- RMSE = typical prediction error **in the units of $y$** ("on average we are off by about …"). Example: residuals
  10, 5, 2, 3, 4 → RMSE $= \sqrt{(100+25+4+9+16)/5} \approx 5.55$.

**Additional explanation.** $R^2$ on the training data never decreases when a feature is added – compare models on test
data or use the adjusted $R^2$. RMSE is easier to communicate, because it is in the units of the target.

In [ ]:
print('R2 =', 1 - 20 / 100)
res = np.array([10, 5, 2, 3, 4])
print('RMSE =', round(np.sqrt(np.mean(res**2)), 2))

## 5. Confidence vs. prediction interval

- **95 % confidence interval (CI):** where the **average** $y$ of all observations with this $x$ lies (uncertainty of
  the estimated line).
- **95 % prediction interval (PI):** where the $y$ of **one new** observation will lie (uncertainty of the line **plus**
  the scatter of single observations).

For simple regression (with $s$ = residual standard error):

$$\text{CI: } \hat y_0 \pm t\cdot s\sqrt{\tfrac{1}{n} + \tfrac{(x_0-\bar x)^2}{\sum (x_i-\bar x)^2}} \qquad
\text{PI: } \hat y_0 \pm t\cdot s\sqrt{1 + \tfrac{1}{n} + \tfrac{(x_0-\bar x)^2}{\sum (x_i-\bar x)^2}}$$

The additional "1" makes the PI always wider. Both intervals become wider far away from $\bar x$. Slide example
(95 m²): CI ≈ 3'000–3'140 CHF, PI ≈ 2'620–3'520 CHF.

In [ ]:
df = pd.DataFrame({'area': x, 'rent': y})
model = smf.ols('rent ~ area', data=df).fit()
grid = pd.DataFrame({'area': np.linspace(25, 145, 100)})
bands = model.get_prediction(grid).summary_frame(alpha=0.05)

plt.figure(figsize=(7, 4.5))
plt.fill_between(grid['area'], bands['obs_ci_lower'], bands['obs_ci_upper'], color='tab:orange', alpha=0.3,
                 label='95% prediction interval')
plt.fill_between(grid['area'], bands['mean_ci_lower'], bands['mean_ci_upper'], color='tab:blue', alpha=0.4,
                 label='95% confidence interval')
plt.plot(grid['area'], bands['mean'], color='tab:blue')
plt.scatter(df['area'], df['rent'], color='black', s=15)
plt.xlabel('living area [m²]')
plt.ylabel('rent [CHF/month]')
plt.legend()
plt.grid(alpha=0.3)
plt.show()
print(model.get_prediction(pd.DataFrame({'area': [95]})).summary_frame(alpha=0.05).round(0))

## 6. Assumptions and diagnostics

| assumption | meaning | check |
|---|---|---|
| linearity | $E[y \mid x]$ is linear in $\beta$ | residuals vs. fitted values |
| independence | observations independent | study design, time order |
| normality | $\varepsilon \sim N(0, \sigma^2)$: needed for exact small-sample t/F inference, **not** for OLS estimation | Q-Q plot |
| equal variance | $\operatorname{Var}(\varepsilon)$ constant (homoscedastic) | residuals vs. fitted values |
| no perfect multicollinearity | perfect: $\beta$ not identifiable; strong: inflated standard errors | correlation matrix, VIF |

The **variance inflation factor** of feature $j$ is $\text{VIF}_j = 1/(1-R_j^2)$, where $R_j^2$ is the $R^2$ of a
regression of $x_j$ on all other features. Rule of thumb: VIF > 5–10 indicates strong multicollinearity.

In [ ]:
rng = np.random.default_rng(2)
fitted = rng.uniform(0, 10, 300)
res_ok = rng.normal(0, 1, 300)
res_het = rng.normal(0, 0.2 + 0.3 * fitted)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].scatter(fitted, res_ok, s=8)
axes[0].set_title('Residuals vs. fitted: OK')
axes[1].scatter(fitted, res_het, s=8, color='tab:orange')
axes[1].set_title('Heteroscedastic: violated')
for ax in axes[:2]:
    ax.axhline(0, color='black', linewidth=1)
    ax.set_xlabel('fitted values')
    ax.set_ylabel('residuals')
sm.qqplot(res_ok, line='45', fit=True, ax=axes[2])
axes[2].set_title('Normal Q-Q plot: OK')
plt.show()

## 7. Multiple predictors and *ceteris paribus*

Slide example (rent explained by rooms and area):

| variable | coefficient | p-value |
|---|---|---|
| intercept | 793.843 | < 0.001 |
| rooms | 36.477 | < 0.001 |
| area [m²] | 9.795 | 0.0045 |

**Ceteris paribus:** $\hat\beta_j$ is the estimated change in $E[y \mid X]$ for +1 in $x_j$, **holding the other
included predictors constant** – an association, not automatically a causal effect. The fitted model is a **plane** in
the (rooms, area, rent) space. Prediction for a flat with 4.5 rooms and 110 m²:
$793.843 + 36.477\cdot 4.5 + 9.795\cdot 110 \approx 2'035$ CHF.

**Additional explanation.** A coefficient in a multiple regression can differ strongly from the slope in a simple
regression with the same feature – even its sign can change – because it answers a different question ("more rooms *at
the same area*" vs. "more rooms").

In [ ]:
print(f'Prediction for 4.5 rooms and 110 m2: {793.843 + 36.477 * 4.5 + 9.795 * 110:.0f} CHF')

## 8. Maximum likelihood (the idea)

**Idea:** choose the parameter values under which the observed data are **most plausible**.

Example: 10 coin tosses with 7 heads and 3 tails. Which probability $p$ of heads? The likelihood of the observed
sequence is $L(p) = p^7 (1-p)^3$:
- $p = 0.5$ → $L = 0.00098$; $p = 0.7$ → $L = 0.00222$ (maximum). The factor 120 for "7 heads in any order" does not
  change the maximum: $\hat p = 0.7$.

**Key idea:** for linear regression with independent normal errors of constant variance, maximum likelihood gives
exactly the OLS line (maximising the normal likelihood = minimising the sum of squared residuals). Logistic regression
is fitted by maximum likelihood.

In [ ]:
p = np.linspace(0, 1, 501)
L = p**7 * (1 - p)**3
print('L(0.5) =', round(0.5**7 * 0.5**3, 5), ' L(0.7) =', round(0.7**7 * 0.3**3, 5))
plt.figure(figsize=(6, 3.5))
plt.plot(p, L)
plt.axvline(p[np.argmax(L)], color='darkred', linestyle='--', label=f'maximum at p = {p[np.argmax(L)]:.2f}')
plt.xlabel('p (probability of heads)')
plt.ylabel('L(p)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 9. Gradient descent

When no closed-form solution exists (or the data are huge), the loss is minimised iteratively. Repeat until the loss
stops decreasing:
1. compute the loss for the current coefficients $\beta$,
2. compute the slope (gradient) of the loss – in which direction does it increase?
3. take a small step of size $\eta$ (learning rate) in the opposite direction: $\beta \leftarrow \beta - \eta\,\nabla L(\beta)$.

Like walking downhill in fog: feel the slope, step down, repeat. Steps that are too large overshoot the valley;
*stochastic* gradient descent uses one observation or a mini-batch per step. For the MSE of simple regression the
gradient is

$$\frac{\partial\,\text{MSE}}{\partial \beta_0} = -\frac{2}{n}\sum_i (y_i - \hat y_i), \qquad
\frac{\partial\,\text{MSE}}{\partial \beta_1} = -\frac{2}{n}\sum_i (y_i - \hat y_i)\,x_i$$

In [ ]:
# Gradient descent on standardised x (makes the loss surface well-conditioned)
z = (x - x.mean()) / x.std()
def run_gd(eta, steps=50):
    b0, b1, path = 0.0, 0.0, []
    for _ in range(steps):
        res = y - (b0 + b1 * z)
        path.append(np.mean(res**2))
        b0 += eta * 2 * np.mean(res)
        b1 += eta * 2 * np.mean(res * z)
    return b0, b1, path

plt.figure(figsize=(7, 4))
for eta in [0.01, 0.1, 0.5, 0.99]:
    b0, b1, path = run_gd(eta)
    plt.plot(path, label=f'learning rate {eta}')
plt.yscale('log')
plt.xlabel('iteration')
plt.ylabel('MSE (log scale)')
plt.title('Gradient descent for simple linear regression')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

b0, b1, _ = run_gd(0.1, steps=500)
ols_slope = np.sum((x - x.mean()) * (y - y.mean())) / np.sum((x - x.mean())**2)
print(f'Slope from gradient descent (back-transformed to m2): {b1 / x.std():.2f}   OLS slope: {ols_slope:.2f}')

**Reading the plot:** with a small learning rate the loss decreases slowly, with a medium one quickly; with a learning
rate close to the stability limit the iterates jump back and forth across the valley. Gradient descent converges to the
same solution as OLS.

**Key messages**
- OLS minimises the sum of squared residuals; the coefficients are interpreted *ceteris paribus*.
- $R^2$ measures the explained share, RMSE the typical error in the units of $y$.
- A prediction interval for one new observation is always wider than the confidence interval for the mean.
- Check the assumptions with residual plots; strong multicollinearity inflates standard errors.

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')